<a href="https://colab.research.google.com/github/KumudithaSilva/llama3-domain-adaptation/blob/feature_fine_tuned_evaluation/llama3_fine_tunined_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama3 Fine-Tuning Model Evaluation

## Import Libraries

In [1]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 42.4 MB/s eta 0:00:00


In [2]:
!wget -q https://raw.githubusercontent.com/KumudithaSilva/llama3-domain-adaptation/feature-base-model/evaluator.py -O evaluator.py

In [7]:
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from evaluator import evaluate

## Load Dataset From HuggingFace

In [4]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

PROJECT_NAME = "stream_price"
TASK = "fine-tuning"

DATA_USER = "KumudithaSilva"
DATASET_NAME = f"{DATA_USER}/stream_items_prompt_lite"

RUN_NAME =  f"{TASK}-{"2026-04-30_11.01.12"}"

FINE_TUNED_MODEL_HUGGINGFACE = "de4a39b1ae7666620aa49a6bff90a97190b098fd"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{DATA_USER}/{PROJECT_RUN_NAME}"

In [5]:
print(f"DATASET_NAME: {DATASET_NAME}")
print(f"RUN_NAME: {RUN_NAME} \n")
print(f"PROJECT_RUN_NAME: {PROJECT_RUN_NAME}")
print(f"HUB_MODEL_NAME: {HUB_MODEL_NAME}")

DATASET_NAME: KumudithaSilva/stream_items_prompt_lite
RUN_NAME: fine-tuning-2026-04-30_11.01.12 

PROJECT_RUN_NAME: stream_price-fine-tuning-2026-04-30_11.01.12
HUB_MODEL_NAME: KumudithaSilva/stream_price-fine-tuning-2026-04-30_11.01.12


## Log in to HuggingFace

In [8]:
hf_token = userdata.get('HUGGING_KEY')
login(hf_token)

## Load Test Dataset From HuggingFace


In [9]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test'].remove_columns(['id'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/563 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/262k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/264k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17600 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2200 [00:00<?, ? examples/s]

## Load Llama Model

### Quantization

In [10]:
# Quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
    )

### Tokenizer

In [11]:
# Tokenizer config
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

### Base Model

In [12]:
# Base model config
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    )

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

## Fine-Tuned Model

In [13]:
fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=FINE_TUNED_MODEL_HUGGINGFACE, device_map="auto")

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/778M [00:00<?, ?B/s]

In [14]:
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

Memory footprint: 2975.7 MB


## Model Prediction

In [15]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]

    return tokenizer.decode(generated_ids)

In [16]:
evaluate(model_predict, test)